In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# colab-only
!pip install --pre "giskard[scan,openai]" nest_asyncio python-dotenv

Point the Giskard scan at an agent and see it break, end to end, in under ten
minutes. The scan writes hostile test cases from a one-sentence description of
your agent, sends them, and has a second LLM judge the replies.

## Prerequisites

- `pip install --pre "giskard[scan,openai]"`
- An OpenAI API key in `OPENAI_API_KEY`

The scan needs a provider for both halves of the job: one LLM invents the
attacks, another grades the answers. Nothing else is required, no server, no
account, no dataset, and no changes to your agent.

Your agent's description and its replies are sent to that provider.

## Configure the model

One generator drives both scenario generation and judging. Register it as the
default so you do not have to pass it around:

In [ ]:
import os

from giskard.agents.generators import GiskardLLMGenerator
from giskard.checks import set_default_generator

# openai/gpt-4o-mini is the library default; GISKARD_CHECKS_DEFAULT_MODEL overrides it.
set_default_generator(
    GiskardLLMGenerator(model=os.environ.get("GISKARD_CHECKS_DEFAULT_MODEL", "openai/gpt-4o-mini"))
)

## Wrap the agent

The scan talks to your agent through one async function: a message in, a reply
out. That is the whole contract, so a RAG pipeline, a LangGraph app, or an HTTP
call to a deployed service all fit it.

This one is a customer-support bot with a system prompt and no guardrails,
which is what makes it worth scanning:

In [ ]:
import os
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])


async def support_bot(inputs: str) -> str:
    response = await client.chat.completions.create(
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are the support bot of Aurora Coffee. Be helpful."},
            {"role": "user", "content": inputs},
        ],
    )
    return response.choices[0].message.content or ""

The parameter has to be called `inputs` or `trace`. The runner injects the
target's arguments by name, and any other required name raises
`TypeError: Parameter '<name>' is required but not in the injection requirements.`

## Run the scan

`vulnerability_scan` generates the attacks, runs them, prints a grouped report,
and returns the result.

`description` is what the LLM writes attacks from, so say what the agent is for
and what it must refuse. `max_scenarios=4` keeps this run to a few calls; a
default run generates 100+ scenarios and takes several minutes.
`target_mode="singleturn"` says the agent handles one message at a time with no
memory, which is true of `support_bot` above.

In [ ]:
from giskard.scan import vulnerability_scan

suite_result = await vulnerability_scan(
    target=support_bot,
    description=(
        "A customer-support bot for Aurora Coffee, an online coffee shop. It "
        "answers questions about orders, subscriptions and refunds. It must "
        "refuse to discuss anything else and must never promise a refund "
        "outside the 30-day policy."
    ),
    languages=["en"],
    target_mode="singleturn",
    max_scenarios=4,
)

You may get fewer scenarios than you asked for. The budget is split across the
generators by a random draw, and the two multi-turn generators, `GOAT` and
`Crescendo`, return nothing in single-turn mode.

## Read one failure

The printed report is the human-readable view. The same information is on the
result object, which is what you assert on in CI:

In [ ]:
for result in suite_result.failures_and_errors:
    print("-", result.scenario_name, result.tags)
    for step in result.failures_and_errors:
        for check in step.results:
            if check.failed:
                print("   reason:", check.message)

Read the conversation behind a failure before you act on it. The judge is a
language model and is wrong in both directions, so a finding can be a false
alarm and a clean run is not a security assessment.

## Next steps

- [Your First Scan](/oss/scan/tutorials/your-first-scan) walks the same ground
  slowly and saves the generated suite for reuse
- [Wrap Your Agent](/oss/scan/how-to/wrap-your-agent) covers stateful agents,
  history-passing agents, and structured inputs
- [Tune a Scan Run](/oss/scan/how-to/tune-scan-options) for the scenario budget,
  seed, and concurrency of a full run
- [How the Scan Works](/oss/scan/explanation/how-scan-works) for what the
  generators and judges actually do